# 06. Prescriptive Generative AI Fine-Tuning (Qwen2.5-7B LoRA)

> **Project:** Smart Manufacturing Maintenance (Predictive & Prescriptive AI)  
> **Foundation Model:** `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`  
> **Adaptation Technique:** QLoRA 4-bit Quantization with Low-Rank Adaptation (Rank $r=16$)  
> **Objective:** Supervised Fine-Tuning (SFT) for Automated Factory Repair SOP & Multi-Failure Safety Guidance.

## 1. Environment & Dependency Setup

In [ ]:
# Install Unsloth and essential fine-tuning dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install gradio

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-e7whcthj/unsloth_c3d93c8171124284be37986d90263298
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-e7whcthj/unsloth_c3d93c8171124284be37986d90263298
  Resolved https://github.com/unslothai/unsloth.git to commit 5a85104f09fb4d753e25a7cec625347554ae1c0e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.8 MB/s eta 0:00:00
   ━━

## 2. Load 4-Bit Quantized Foundation Model & Configure LoRA

Configuring `Qwen2.5-7B-Instruct` with 4-bit NormalFloat quantization and injecting LoRA adapters across attention and MLP projection layers.

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None         # Auto-detects float16/bfloat16 hardware support
load_in_4bit = True  # 4-bit memory quantization

# 1. Load Base Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Inject LoRA Adapters for Target Modules
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✓ Base model loaded with 4-bit quantization and LoRA adapters successfully configured.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Industrial SOP Dataset Preprocessing (Alpaca Instruction Format)

Structuring synthetic manufacturing dataset (`sop_dataset.jsonl`) into standardized instruction prompt format.

In [ ]:
from datasets import load_dataset

# Standard Alpaca Prompt Template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Load and tokenize dataset
dataset = load_dataset("json", data_files="sop_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✓ Loaded and formatted {len(dataset)} instruction-response pairs.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

## 4. Supervised Fine-Tuning (SFT) Execution

Executing parameter-efficient fine-tuning with `SFTTrainer`, AdamW 8-bit optimizer, and linear learning rate schedule.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 25,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start SFT Training Process
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/140 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 140 | Num Epochs = 2 | Total steps = 25
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.318232
2,2.323049
3,2.227355
4,2.132784
5,2.026142
6,1.910681
7,1.795157
8,1.674597
9,1.587816
10,1.479494


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-25/tokenizer_config.json.


## 5. Model Adapter Serialization & Export

Saving the lightweight LoRA adapter weights and tokenizer configuration for local inference.

In [ ]:
# Save LoRA Adapter Weights and Tokenizer
export_folder = "qwen-sop-model"
model.save_pretrained(export_folder)
tokenizer.save_pretrained(export_folder)

print(f"✓ LoRA adapter weights saved successfully to folder: '{export_folder}/'")

Unsloth: Restored added_tokens_decoder metadata in qwen-sop-model/tokenizer_config.json.


Model berhasil disimpan di folder 'qwen-sop-model'!


## 6. Interactive Prescriptive Assistant Demo (Gradio Interface)

Instantiating local inference with temperature control and system prompt grounding for industrial SOP generation.

In [ ]:
import gradio as gr
import torch
from unsloth import FastLanguageModel

# 1. Prepare Model for Fast Inference
FastLanguageModel.for_inference(model)

# 2. Define Diagnostic Inference Engine
def chat_with_ai(message, history):
    with torch.no_grad():
        messages = [
            {
                "role": "system",
                "content": (
                    "Kamu adalah asisten teknisi ahli di pabrik manufaktur. "
                    "Berikan laporan diagnostik lengkap (Akar Masalah, Konteks Varian Mesin, "
                    "SOP Perbaikan, dan Pencegahan) berdasarkan laporan deteksi kerusakan mesin."
                )
            }
        ]

        # Parse chat history
        for item in history:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                messages.append({"role": "user", "content": str(item[0])})
                if item[1] is not None:
                    messages.append({"role": "assistant", "content": str(item[1])})
            elif isinstance(item, dict):
                messages.append({
                    "role": str(item.get("role", "user")),
                    "content": str(item.get("content", ""))
                })

        # Parse current prompt
        user_text = str(message.get("text", message.get("content", ""))) if isinstance(message, dict) else str(message)
        messages.append({"role": "user", "content": user_text})

        # Tokenize and format prompt
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt",
        ).to("cuda")

        attention_mask = torch.ones_like(inputs)

        # Generate response
        outputs = model.generate(
            input_ids = inputs,
            attention_mask = attention_mask,
            pad_token_id = tokenizer.eos_token_id,
            max_new_tokens = 1536,
            use_cache = True,
            temperature = 0.3,
        )

        response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
        return response

# 3. Build Gradio Chat Interface
demo = gr.ChatInterface(
    fn = chat_with_ai,
    title = "Maintain: Asisten Prescriptive Maintenance AI",
    description = "Masukkan laporan sensor dari Predictive AI, atau tanyakan detail panduan perbaikan mesin kepada asisten.",
    examples = [
        "Laporan Deteksi AI: Mesin L47181 terdeteksi mengalami OSF. Data Sensor -> Suhu Udara: 298K, Suhu Proses: 308K, Kecepatan: 1300 rpm, Torsi: 68 Nm, Keausan Alat: 210 menit.",
        "Laporan Deteksi AI: Mesin H47182 terdeteksi mengalami HDF dan TWF secara bersamaan. Data Sensor -> Suhu Udara: 300K, Suhu Proses: 312K, Kecepatan: 1350 rpm, Torsi: 45 Nm, Keausan Alat: 215 menit."
    ]
)

# 4. Launch Local Web Interface
demo.launch(share=True, debug=False)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Sedang memuat model...
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.19 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Menyiapkan Antarmuka Chatbot...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7df342b5db76195e9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=1536) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1536) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
